# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

The original step-by-step plan lives in `rpo_legacy.ipynb`.

In [139]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [140]:
import numpy as np
from cesium.constants import DAY_TO_SEC, DEG_TO_RAD, KM_TO_M, M_TO_KM, MIN_TO_SEC, RAD_TO_DEG
from cesium.guidance import RpoTimeline
from cesium.guidance.coplanar import guid_coaxial_hohmann, guid_phasing_same_orbit
from cesium.guidance.hcw import hcw_two_impulse, run_rpo
from cesium.guidance.noncoplanar import guid_combined_i_raan_change, node_line_finder, guid_plane_change_3d
from cesium.math.base import unit
from cesium.math.rotations import wrap_to_2pi, wrap_to_pi
from cesium.math.time import jd_to_greg, jd_to_sec_j2000
from cesium.orbital.coes import COES, orbit_sweep
from cesium.plotting import plot_3d, plot_rsw, plot_separation
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from numpy.linalg import norm

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [141]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_0

dt = 1  # sec

In [142]:
# Chaser = target + seeded uniform dispersion
SEED = 525600
# SEED = 3214359876
# SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
J2 = 0
INITIAL_TA_OFFSET_DEG = 57

# CRAZY
# ANGLE_DISP_DEG = 50.0  # i, raan, aop
# A_DISP_M = 500 * KM_TO_M
J2 = J2_MARS

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_0 = COES.from_array(TARGET_0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_0

TARGET_0.print("initial target coes")
CHASER_0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3754.914 km, e = 0.00763, i = 46.710 deg, raan = 15.208 deg, aop = 2.930 deg, ta = 70.000 deg


In [143]:
# From arrival.ipynb
MISSION_NAME = "birthday-christmas-earlier"
mission = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)
EPOCH0_JD_TDB = float(mission["lambert_arrival_jd"])
EPOCH0_TDB_S = jd_to_sec_j2000(EPOCH0_JD_TDB)

print(
    "RPO start: {:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f} TDB".format(
        *jd_to_greg(EPOCH0_JD_TDB)
    ),
    f"({EPOCH0_TDB_S:.3f} s past J2000)",
)

RPO start: 2035-12-25 00:00:00.000 TDB (1135425600.000 s past J2000)


## Burn timeline
One propagated truth for both spacecraft (`cesium.guidance.RpoTimeline`): every `coast` continues from the last state, every `burn` is applied to it, and each burn is planned from the current (propagated) COES. Times are seconds since `EPOCH0_TDB_S`; `tl.epochs_tdb_s()` and `tl.burn_table()` give absolute epochs.

- Tried hohmann corrections

In [144]:
def raan_dot_j2(a, e, i):
    """Secular J2 nodal regression rate [rad/s]."""
    n = np.sqrt(MU_MARS / a**3)
    p = a * (1 - e**2)
    return -1.5 * n * J2 * (R_MARS / p) ** 2 * np.cos(i)

In [159]:
K_PHASE = 20
HCW_THRESH = 20 * KM_TO_M
DOCK_THRESH = 1 * KM_TO_M
T_RENDEZVOUS = 40 * MIN_TO_SEC

# Pass 1: no bias, to measure the J2 differential nodal drift accumulated during phasing
tl_0, info_0 = run_rpo(CHASER_0, TARGET_0, K_PHASE, HCW_THRESH, DOCK_THRESH, T_RENDEZVOUS,
                       raan_bias=0.0, verbose=True)

# Analytic estimate: phasing orbit sits higher, so its node regresses more slowly
drift_analytic = (
    raan_dot_j2(info_0["a_phase"], 0, info_0["i_target"])
    - raan_dot_j2(info_0["a_target"], 0, info_0["i_target"])
) * info_0["t_phase"]
print(
    f"RAAN drift during phasing: measured {info_0['draan'] * RAD_TO_DEG:.4f} deg, "
    f"analytic {drift_analytic * RAD_TO_DEG:.4f} deg\n"
)

# Pass 2: pre-bias Burn 1 by the measured drift
raan_bias = -info_0["draan"]
print(f"Re-running with RAAN bias {raan_bias * RAD_TO_DEG:.4f} deg")
tl, info = run_rpo(CHASER_0, TARGET_0, K_PHASE, HCW_THRESH, DOCK_THRESH, T_RENDEZVOUS,
                   raan_bias=raan_bias, verbose=True)
n_hat = info["n_hat"]

print(f"\nJ2 bias saves {tl_0.dv_total - tl.dv_total:.3f} m/s")

ts_tl, rv_c_tl, rv_t_tl = tl.arrays()
fig_sep = plot_separation(ts_tl, rv_c_tl, rv_t_tl, tl.burn_events())

Phasing exit: phases_remaining=0, sep 12.211 km (R -3.701, S 6.497, W 9.654 km), dRAAN 0.2161 deg
HCW exit: sep 0.217 km (R -0.114, S 0.156, W -0.098 km), dRAAN 0.2161 deg
total dv = 164.472 m/s
RAAN drift during phasing: measured 0.2161 deg, analytic 0.2057 deg

Re-running with RAAN bias -0.2161 deg
Phasing exit: phases_remaining=0, sep 5.476 km (R -3.741, S -3.857, W 1.058 km), dRAAN -0.0105 deg
HCW exit: sep 0.209 km (R -0.130, S 0.164, W 0.004 km), dRAAN -0.0105 deg
total dv = 151.773 m/s

J2 bias saves 12.699 m/s


### HCW Targeting

In [146]:
print(tl.burn_table())

for b in tl.burns:
    if b.name.startswith("HCW") and b.name.endswith("B"):
        rr = tl.rel_at(b.t_s)
        print(f"after {b.name}: |r| = {norm(rr[0:3]):8.2f} m, |v| = {norm(rr[3:6]):.4f} m/s")

  #     t [hr]           JD (TDB)  |dv| [m/s]  name
  0      0.609     2451545.025354     106.602  Burn 1 (plane)
  1      1.579     2451545.065799      24.471  Burn 2 (Hohmann)
  2      2.532     2451545.105497       0.286  Burn 3 (circularize + phase entry)
  3      4.437     2451545.184879       0.005  Burn 4.0 (phasing)
  4      6.342     2451545.264259       0.371  Burn 4.1 (phasing)
  5      8.248     2451545.343663       0.366  Burn 4.2 (phasing)
  6     10.154     2451545.423090       0.354  Burn 4.3 (phasing)
  7     12.061     2451545.502539       0.334  Burn 4.4 (phasing)
  8     13.968     2451545.582007       0.307  Burn 4.5 (phasing)
  9     15.876     2451545.661494       0.271  Burn 4.6 (phasing)
 10     17.784     2451545.740995       0.229  Burn 4.7 (phasing)
 11     19.692     2451545.820509       0.180  Burn 4.8 (phasing)
 12     21.601     2451545.900030       0.126  Burn 4.9 (phasing)
 13     23.509     2451545.979555       0.068  Burn 4.10 (phasing)
 14     25.41

In [155]:
ts_rsw, rv_rsw = tl.rsw_arrays(progress=True)
burns_tl = tl.burn_events()
plot_rsw(ts_rsw[::5], rv_rsw[::5], burns_tl, show=False)

# Zoom on phasing: everything from burn 3 on
# m = ts_rsw >= tl.burns[2].t_s
# plot_rsw(ts_rsw[m], rv_rsw[m], burns_tl[2:])

100%|██████████| 25975/25975 [00:01<00:00, 25649.66it/s]


In [148]:
# 3D view of the chaser in the target's RSW frame (plot X/Y/Z = R/S/W), target at the origin
# Zoom on HCW: m = ts_rsw >= next(t_b for t_b, name in burns_tl if name.startswith("HCW"))
m = np.ones_like(ts_rsw, dtype=bool)
r_rel_rsw = rv_rsw[m, 0:3]
ts_rel = ts_rsw[m]

axis_len = 0.01 * norm(r_rel_rsw, axis=1).max()  # [m]
rsw_axes = [
    {
        "direction": [1, 0, 0],
        "length": axis_len,
        "name": "R",
        "color": "red",
        "label": True,
    },
    {
        "direction": [0, 1, 0],
        "length": axis_len,
        "name": "S",
        "color": "green",
        "label": True,
    },
    {
        "direction": [0, 0, 1],
        "length": axis_len,
        "name": "W",
        "color": "blue",
        "label": True,
    },
]

burn_points = [
    {"name": "Target", "location": np.zeros(3), "color": "black", "symbol": "x"}
]
for t_b, name in burns_tl:
    if t_b >= ts_rel[0]:
        k = np.argmin(np.abs(ts_rel - t_b))
        burn_points.append(
            {"name": name, "location": r_rel_rsw[k], "color": "orange", "size": 4}
        )
burn_points = []
fig_rsw_3d = plot_3d(
    r_rel_rsw[-10000::5],
    ts_rel[-10000::5],
    labels="Chaser (rel. target)",
    trace_colors="royalblue",
    title="Chaser relative to target (RSW: X = R, Y = S, Z = W)",
    vectors=rsw_axes,
    points=burn_points,
    scale=KM_TO_M,  # inputs in m, displayed in km
    units="km",
)

In [149]:
ts_tl, rv_c_tl, rv_t_tl = tl.arrays()
fig_sep = plot_separation(ts_tl, rv_c_tl, rv_t_tl, tl.burn_events())

## Visualization

In [150]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

fig = plot_3d(
    [rv_t_tl[:, 0:3] * M_TO_KM, rv_c_tl[:, 0:3] * M_TO_KM],
    [ts_tl, ts_tl],
    labels=["Target", "Chaser"],
    trace_colors=["black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title="Target and chaser, full timeline (MCI)",
    vectors=vectors,
    units="km",
)

# end